# 02 · Embeddings: search by meaning

**Module 03 · Agent Memory & Retrieval**

A shop has many policy documents. A customer asks, "How do I get my money back?". The right answer is in a paragraph about **refunds**, which does not contain the words "money back".

A normal text search (matching words) would miss it. **Embeddings** let us search by **meaning**.

An embedding is a **list of numbers that captures what a text means**. Texts with similar meanings get similar numbers, so we can measure how close two texts are.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   # read your keys from the .env file
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to your .env file (see .env.example)"

client = OpenAI()        # finds OPENAI_API_KEY on its own
MODEL = "gpt-4o-mini"    # a small, cheap model is plenty for this module
print("Ready")

Ready


## One text, one vector
We ask OpenAI's embedding model to turn a sentence into numbers.

In [2]:
def embed(texts):
    """Turn a list of texts into a list of vectors (a vector is just a long list of numbers)."""
    response = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return [item.embedding for item in response.data]

def similarity(vector_a, vector_b):
    """How close in meaning are two texts? Higher = closer.
    OpenAI vectors all have length 1, so multiplying them pair by pair and adding up is enough."""
    return sum(a * b for a, b in zip(vector_a, vector_b))

In [3]:
vector = embed(["How do I get my money back?"])[0]

print("Numbers in the vector:", len(vector))
print("First five numbers   :", [round(n, 4) for n in vector[:5]])

Numbers in the vector: 1536
First five numbers   : [-0.0046, -0.0173, -0.0127, 0.0413, -0.0556]


That is 1,536 numbers for one sentence. You do not read them. You only **compare** them.

## Which texts are closest in meaning?
We compare one customer question against three policy sentences. The `similarity` function gives a score. **Higher means closer in meaning.**

In [4]:
question = "How do I get my money back?"

sentences = [
    "Refunds go back to the original payment method.",
    "Standard shipping takes 3 to 5 business days.",
    "The warranty covers manufacturing faults.",
]

question_vector = embed([question])[0]
sentence_vectors = embed(sentences)

print(f"Question: {question}\n")
for sentence, vector in zip(sentences, sentence_vectors):
    print(f"  {similarity(question_vector, vector):.2f}  {sentence}")

Question: How do I get my money back?

  0.48  Refunds go back to the original payment method.
  0.12  Standard shipping takes 3 to 5 business days.
  0.12  The warranty covers manufacturing faults.


The refund sentence scores highest **even though it shares no important words with the question**. The model understands that "get my money back" and "refunds" mean the same thing.

## A tiny search engine
Searching is just: embed the question, score every text, sort, take the best.

In [5]:
def search(question, texts, top_k=2):
    question_vector = embed([question])[0]
    text_vectors = embed(texts)

    scored = [(similarity(question_vector, v), text) for v, text in zip(text_vectors, texts)]
    scored.sort(reverse=True)           # best match first
    return scored[:top_k]

policies = [
    "Refunds go back to the original payment method.",
    "Standard shipping takes 3 to 5 business days.",
    "Express shipping costs $12.",
    "The warranty covers manufacturing faults.",
    "You can cancel an order for free until it ships.",
]

for question in ["Can I still stop my order?", "How quick is delivery?"]:
    print(f"\nQuestion: {question}")
    for score, text in search(question, policies):
        print(f"  {score:.2f}  {text}")


Question: Can I still stop my order?


  0.59  You can cancel an order for free until it ships.
  0.27  Refunds go back to the original payment method.

Question: How quick is delivery?


  0.38  Standard shipping takes 3 to 5 business days.
  0.35  Express shipping costs $12.


Both questions find the right policy, with different words from the policy text.

## Recap
* An **embedding** turns text into numbers that capture its meaning.
* Close meaning means a **high similarity score**.
* Searching by meaning is: embed, compare, sort, take the top few.
* Embeddings are very cheap, a fraction of a cent for thousands of sentences.

Next: use this search to let the model **answer from your documents**. That is RAG.